<a href="https://colab.research.google.com/github/boba1987/advanced-neural-networks-and-deep-learning/blob/main/Customer_Support_Ticket_Classifier_BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# Customer Support Ticket Classification — BERT

Fine-tune **`bert-base-uncased`** to predict ticket **type** (`Incident`, `Request`, `Problem`, `Change`) from **`body` text only**.

Uses the same dataset and train/validation/test split as [`Customer_Support_Ticket_Classifier.ipynb`](Customer_Support_Ticket_Classifier.ipynb) (TF-IDF + MLP baseline).

**Dataset:** `dataset-tickets-en.csv` (~11,921 rows) | **Split:** 80% train / 10% val / 20% test


## Colab notes

- **Runtime → Change runtime type → T4 GPU** before running.
- Data is fetched with **`curl`** from GitHub.
- Compare results with the MLP notebook on the same held-out test set.


## Phase 1: Setup and data loading


In [ ]:
!pip install -q torch transformers scikit-learn pandas matplotlib seaborn joblib


In [ ]:
import random
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cpu":
    print("Warning: BERT fine-tuning is very slow on CPU. Enable a GPU runtime in Colab.")


### Load dataset


In [ ]:
GITHUB_RAW_URL = "https://raw.githubusercontent.com/boba1987/advanced-neural-networks-and-deep-learning/main/dataset-tickets-en.csv"
CSV_PATH = "dataset-tickets-en.csv"

!curl -L -o {CSV_PATH} "{GITHUB_RAW_URL}"

raw = pd.read_csv(CSV_PATH, usecols=["body", "type"])
raw["body"] = raw["body"].astype(str).str.strip()
raw["type"] = raw["type"].astype(str).str.strip()

df = pd.DataFrame({
    "ticket_text": raw["body"],
    "category": raw["type"],
})
df = df[(df["ticket_text"].str.len() > 0) & (df["category"].str.len() > 0)].drop_duplicates(subset="ticket_text").reset_index(drop=True)

print(f"Loaded {len(df):,} tickets")
print(f"Categories: {sorted(df['category'].unique())}")
print(df["category"].value_counts())


## Phase 2: Preprocessing, labels, and split

Same `random_state=42` stratified split as the MLP notebook.


In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower().strip()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["text_clean"] = df["ticket_text"].apply(clean_text)

label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["category"])
num_classes = len(label_encoder.classes_)
id_to_label = {i: label_encoder.classes_[i] for i in range(num_classes)}
print("Label mapping:", id_to_label)

texts_raw = df["ticket_text"].values
texts_clean = df["text_clean"].values
labels = df["label"].values

X_train_full, X_test_orig, y_train_full, y_test, texts_train_full, texts_test = train_test_split(
    texts_raw, labels, texts_clean,
    test_size=0.2, random_state=42, stratify=labels,
)

X_train, X_val, y_train, y_val, texts_train, texts_val = train_test_split(
    X_train_full, y_train_full, texts_train_full,
    test_size=0.1, random_state=42, stratify=y_train_full,
)

print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test_orig):,}")


## Phase 3: BERT fine-tuning


In [ ]:
# Re-detect device in case GPU was enabled after the setup cell.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin_memory = device.type == "cuda"

BERT_MODEL_NAME = "bert-base-uncased"
MAX_LEN = 256
BATCH_SIZE = 16 if device.type == "cuda" else 8
MAX_EPOCHS = 3
BERT_LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
EARLY_STOP_PATIENCE = 2

class_weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL_NAME, num_labels=num_classes
).to(device)

class TicketTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels.astype(np.int64)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_ds = TicketTextDataset(texts_train, y_train, tokenizer, MAX_LEN)
val_ds = TicketTextDataset(texts_val, y_val, tokenizer, MAX_LEN)
test_ds = TicketTextDataset(texts_test, y_test, tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=pin_memory)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=pin_memory)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=pin_memory)

loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.AdamW(model.parameters(), lr=BERT_LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * MAX_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

history = {"train_loss": [], "train_acc": [], "train_f1": [], "val_loss": [], "val_acc": [], "val_f1": []}
best_val_f1 = 0.0
best_val_acc = 0.0
best_epoch = 0
epochs_no_improve = 0
best_state = None

print(f"Model: {BERT_MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Batch size: {BATCH_SIZE} | Max length: {MAX_LEN} | Device: {device}")


In [ ]:
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels_batch = batch["labels"].to(device)

        if train:
            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits, labels_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
        else:
            with torch.no_grad():
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = loss_fn(outputs.logits, labels_batch)

        preds = outputs.logits.argmax(dim=1)
        total_loss += loss.item() * labels_batch.size(0)
        correct += (preds == labels_batch).sum().item()
        total += labels_batch.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels_batch.cpu().numpy())

    acc = correct / total
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return total_loss / total, acc, macro_f1

print(f"Training BERT on {device}...\n")
for epoch in range(MAX_EPOCHS):
    train_loss, train_acc, train_f1 = run_epoch(train_loader, train=True)
    val_loss, val_acc, val_f1 = run_epoch(val_loader, train=False)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["train_f1"].append(train_f1)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_f1"].append(val_f1)

    print(
        f"Epoch [{epoch+1:2d}/{MAX_EPOCHS}] | "
        f"train loss {train_loss:.4f} acc {train_acc*100:.2f}% F1 {train_f1*100:.2f}% | "
        f"val loss {val_loss:.4f} acc {val_acc*100:.2f}% F1 {val_f1*100:.2f}% | "
        f"lr {optimizer.param_groups[0]['lr']:.6f}"
    )

    if val_f1 > best_val_f1:
        best_val_f1, best_val_acc, best_epoch, epochs_no_improve = val_f1, val_acc, epoch + 1, 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

if best_state is not None:
    model.load_state_dict(best_state)
    model.to(device)

print(f"\nBest epoch {best_epoch}, val acc {best_val_acc*100:.2f}%, val macro F1 {best_val_f1*100:.2f}%")


## Phase 4: Evaluation


In [ ]:
def predict_loader(loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels_batch = batch["labels"]
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            probs = torch.softmax(logits, dim=1)
            preds = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels_batch.numpy())
            all_probs.append(probs.cpu().numpy())
    return np.array(all_preds), np.array(all_labels), np.vstack(all_probs)

y_pred, y_true, y_probs = predict_loader(test_loader)
test_acc = accuracy_score(y_true, y_pred)
test_f1 = f1_score(y_true, y_pred, average="macro")
top_k = min(3, num_classes)
topk_acc = np.mean([y_true[i] in np.argsort(y_probs[i])[-top_k:] for i in range(len(y_true))])

print(f"Test accuracy: {test_acc*100:.2f}%")
print(f"Test macro F1: {test_f1*100:.2f}%")
print(f"Top-{top_k} accuracy: {topk_acc*100:.2f}%")
print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=label_encoder.classes_))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
ep = range(1, len(history["train_loss"]) + 1)
axes[0].plot(ep, history["train_loss"], label="Train")
axes[0].plot(ep, history["val_loss"], label="Val")
axes[0].set_title("Loss")
axes[1].plot(ep, [a * 100 for a in history["train_acc"]], label="Train")
axes[1].plot(ep, [a * 100 for a in history["val_acc"]], label="Val")
axes[1].set_title("Accuracy (%)")
axes[2].plot(ep, [a * 100 for a in history["train_f1"]], label="Train")
axes[2].plot(ep, [a * 100 for a in history["val_f1"]], label="Val")
axes[2].set_title("Macro F1 (%)")
for ax in axes:
    ax.legend()
plt.suptitle("Training curves — BERT")
plt.tight_layout()
plt.show()

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Greens",
    xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_,
)
plt.title("Confusion matrix — BERT")
plt.ylabel("True")
plt.xlabel("Predicted")
plt.tight_layout()
plt.show()


## Phase 5: Save and inference


In [ ]:
def predict_ticket(description, top_k=None):
    """Predict ticket type from body text using fine-tuned BERT."""
    if top_k is None:
        top_k = num_classes
    model.eval()
    clean = clean_text(description)
    encoding = tokenizer(
        clean,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
        return_tensors="pt",
    )
    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)
    with torch.no_grad():
        probs = torch.softmax(
            model(input_ids=input_ids, attention_mask=attention_mask).logits, dim=1
        ).cpu().numpy()[0]
    top_idx = np.argsort(probs)[-top_k:][::-1]
    snippet = description[:200] + ("..." if len(description) > 200 else "")
    print(snippet)
    print("-" * 50)
    for rank, idx in enumerate(top_idx, 1):
        print(f"{rank}. {id_to_label[idx]:<35} {probs[idx]*100:.2f}%")
    print()

predict_ticket("The analytics platform crashed and is down — we need it restored immediately.")
predict_ticket("Can you provide documentation on how to configure two-factor authentication?")
predict_ticket("Recurring sync errors between systems after the last software update.")
predict_ticket("We plan to migrate the database next month and need approval for the change window.")


In [ ]:
SAVE_DIR = "bert_ticket_classifier"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
joblib.dump(label_encoder, f"{SAVE_DIR}/label_encoder.pkl")
print(f"Saved model, tokenizer, and label encoder to {SAVE_DIR}/")
